# GO2 Velocity Error Polynomial Model (OptiTrack vs Measured)

This notebook fits polynomial models for:

- `e_vx = vx_true - vx_measured`
- `e_vy = vy_true - vy_measured`

`vx_true` and `vy_true` are derived from OptiTrack world-frame velocities, then rotated to robot frame using heading:

`vx_robot = cos(heading)*vx_world + sin(heading)*vy_world`  
`vy_robot = -sin(heading)*vx_world + cos(heading)*vy_world`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

# Config
DATASET_CANDIDATES = [
    Path("go2/dataset"),
    Path("examples/go2_imu_model/go2/dataset"),
]
POLY_DEGREE = 2
RIDGE_ALPHA = 1e-3
DT_MAX_SEC = 0.2
VEL_CLIP_MS = 5.0
VYAW_CLIP = 10.0
ACC_CLIP_MS2 = 20.0

REQUIRED_COLUMNS = {
    "sample_ts_ns",
    "sport_heading",
    "sport_vx",
    "sport_vy",
    "sport_vyaw",
    "rigid_x",
    "rigid_y",
}


def resolve_dataset_dir() -> Path:
    for d in DATASET_CANDIDATES:
        if d.exists():
            return d
    raise FileNotFoundError(
        "Could not find dataset directory. Checked: " + ", ".join(str(d) for d in DATASET_CANDIDATES)
    )


DATASET_DIR = resolve_dataset_dir()
print(f"Using dataset dir: {DATASET_DIR}")


In [ ]:
def world_to_body(vx_world: np.ndarray, vy_world: np.ndarray, heading: np.ndarray):
    c = np.cos(heading)
    s = np.sin(heading)
    vx_body = c * vx_world + s * vy_world
    vy_body = -s * vx_world + c * vy_world
    return vx_body, vy_body


def process_recording(csv_path: Path) -> pd.DataFrame | None:
    df = pd.read_csv(csv_path)
    if len(df) < 5:
        return None
    if not REQUIRED_COLUMNS.issubset(df.columns):
        return None

    t = df["sample_ts_ns"].to_numpy(dtype=float) / 1e9
    t = t - t[0]

    imu_heading = df["sport_heading"].to_numpy(dtype=float)
    imu_vx = df["sport_vx"].to_numpy(dtype=float)
    imu_vy = df["sport_vy"].to_numpy(dtype=float)
    imu_vyaw = df["sport_vyaw"].to_numpy(dtype=float)

    # OptiTrack position projected to the ground plane in the same convention
    # used by existing scripts: opti_x=-rigid_y, opti_y=rigid_x.
    opti_x = -df["rigid_y"].to_numpy(dtype=float)
    opti_y = df["rigid_x"].to_numpy(dtype=float)

    # World-frame velocities from finite differences.
    opti_vx_world = np.gradient(opti_x, t)
    opti_vy_world = np.gradient(opti_y, t)

    # Rotate world-frame OptiTrack velocity to robot frame (same logic for vx, vy).
    true_vx, true_vy = world_to_body(opti_vx_world, opti_vy_world, imu_heading)

    # Measured accelerations from IMU linear velocities.
    imu_ax = np.gradient(imu_vx, t)
    imu_ay = np.gradient(imu_vy, t)
    imu_acc = np.hypot(imu_ax, imu_ay)

    # Targets: velocity error.
    err_vx = true_vx - imu_vx
    err_vy = true_vy - imu_vy

    out = pd.DataFrame({
        "t": t,
        "dt": np.r_[np.nan, np.diff(t)],
        "imu_vx": imu_vx,
        "imu_vy": imu_vy,
        "imu_vyaw": imu_vyaw,
        "imu_ax": imu_ax,
        "imu_ay": imu_ay,
        "imu_acc": imu_acc,
        "true_vx": true_vx,
        "true_vy": true_vy,
        "err_vx": err_vx,
        "err_vy": err_vy,
        "recording": csv_path.stem,
    })
    return out

In [ ]:
csv_files = sorted(DATASET_DIR.glob("data_rigid_bodies*.csv"))
print(f"Found {len(csv_files)} candidate files in {DATASET_DIR}")

parts = []
for path in csv_files:
    seg = process_recording(path)
    if seg is None:
        print(f"[skip] {path.name}")
        continue
    parts.append(seg)
    print(f"[ok]   {path.name:55s} rows={len(seg):4d}")

if not parts:
    raise RuntimeError("No valid data_rigid_bodies*.csv files found with required columns")

raw = pd.concat(parts, ignore_index=True)
print(f"\nTotal rows before cleaning: {len(raw)}")


In [ ]:
mask = (
    np.isfinite(raw[[
        "dt", "imu_vx", "imu_vy", "imu_vyaw", "imu_ax", "imu_ay", "imu_acc",
        "true_vx", "true_vy", "err_vx", "err_vy"
    ]]).all(axis=1)
    & (raw["dt"] > 1e-5)
    & (raw["dt"] < DT_MAX_SEC)
    & (raw["true_vx"].abs() < VEL_CLIP_MS)
    & (raw["true_vy"].abs() < VEL_CLIP_MS)
    & (raw["imu_vyaw"].abs() < VYAW_CLIP)
    & (raw["imu_ax"].abs() < ACC_CLIP_MS2)
    & (raw["imu_ay"].abs() < ACC_CLIP_MS2)
)

data = raw.loc[mask].copy().reset_index(drop=True)
print(f"Retained rows: {len(data)} / {len(raw)}")
print(f"Recordings: {data['recording'].nunique()}")

data[[
    "imu_vx", "imu_vy", "imu_vyaw", "imu_ax", "imu_ay", "imu_acc", "err_vx", "err_vy"
]].describe().round(5)

## Polynomial Fit Formulation

We fit two polynomial regressors on velocity error:

- `e_vx = v_{x,true} - v_{x,measured}`
- `e_vy = v_{y,true} - v_{y,measured}`

Feature vector:
`x = [v_x, v_y, \omega_z, a_x, a_y, |a|]`

Each model is:
`e(x) = \phi(x)^T w + b`
where `\phi(x)` contains all polynomial terms up to degree 2, and `w,b` are learned by Ridge regression.

Recovered velocity is then:
- `v_{x,pred} = v_{x,measured} + \hat e_{vx}(x)`
- `v_{y,pred} = v_{y,measured} + \hat e_{vy}(x)`


In [ ]:
feature_cols = ["imu_vx", "imu_vy", "imu_vyaw", "imu_ax", "imu_ay", "imu_acc"]
X = data[feature_cols].to_numpy()

y_vx = data["err_vx"].to_numpy()
y_vy = data["err_vy"].to_numpy()

model_vx = make_pipeline(
    PolynomialFeatures(degree=POLY_DEGREE, include_bias=True),
    Ridge(alpha=RIDGE_ALPHA),
)
model_vy = make_pipeline(
    PolynomialFeatures(degree=POLY_DEGREE, include_bias=True),
    Ridge(alpha=RIDGE_ALPHA),
)

model_vx.fit(X, y_vx)
model_vy.fit(X, y_vy)

pred_err_vx = model_vx.predict(X)
pred_err_vy = model_vy.predict(X)

print(f"Polynomial degree: {POLY_DEGREE}")
print(f"Input features: {feature_cols}")
print("\nIn-sample fit quality")
print(f"err_vx R^2: {r2_score(y_vx, pred_err_vx):.4f}")
print(f"err_vy R^2: {r2_score(y_vy, pred_err_vy):.4f}")


In [ ]:
# Compare raw measured velocity vs corrected velocity.
true_vx = data["true_vx"].to_numpy()
true_vy = data["true_vy"].to_numpy()
imu_vx = data["imu_vx"].to_numpy()
imu_vy = data["imu_vy"].to_numpy()

corr_vx = imu_vx + pred_err_vx
corr_vy = imu_vy + pred_err_vy

rmse_raw_vx = np.sqrt(mean_squared_error(true_vx, imu_vx))
rmse_cor_vx = np.sqrt(mean_squared_error(true_vx, corr_vx))
rmse_raw_vy = np.sqrt(mean_squared_error(true_vy, imu_vy))
rmse_cor_vy = np.sqrt(mean_squared_error(true_vy, corr_vy))

print("Velocity RMSE against OptiTrack-derived truth")
print(f"vx raw      : {rmse_raw_vx:.5f} m/s")
print(f"vx corrected: {rmse_cor_vx:.5f} m/s")
print(f"vy raw      : {rmse_raw_vy:.5f} m/s")
print(f"vy corrected: {rmse_cor_vy:.5f} m/s")

In [ ]:
# Group-wise cross-validation by recording to avoid leakage.
groups = pd.factorize(data["recording"])[0]
n_groups = len(np.unique(groups))
n_splits = min(5, n_groups)
gkf = GroupKFold(n_splits=n_splits)

vx_cv = cross_val_score(
    make_pipeline(PolynomialFeatures(POLY_DEGREE, include_bias=True), Ridge(alpha=RIDGE_ALPHA)),
    X,
    y_vx,
    groups=groups,
    cv=gkf,
    scoring="r2",
)
vy_cv = cross_val_score(
    make_pipeline(PolynomialFeatures(POLY_DEGREE, include_bias=True), Ridge(alpha=RIDGE_ALPHA)),
    X,
    y_vy,
    groups=groups,
    cv=gkf,
    scoring="r2",
)

print(f"GroupKFold over {n_groups} recordings ({n_splits} splits)")
print(f"err_vx CV R^2: {vx_cv.mean():.4f} ± {vx_cv.std():.4f}")
print(f"err_vy CV R^2: {vy_cv.mean():.4f} ± {vy_cv.std():.4f}")

In [ ]:
feat_names = (
    PolynomialFeatures(degree=POLY_DEGREE, include_bias=True)
    .fit(X)
    .get_feature_names_out(feature_cols)
)

coef_vx = model_vx.named_steps["ridge"].coef_
coef_vy = model_vy.named_steps["ridge"].coef_
int_vx = model_vx.named_steps["ridge"].intercept_
int_vy = model_vy.named_steps["ridge"].intercept_

coef_tbl = pd.DataFrame({
    "term": feat_names,
    "coef_err_vx": coef_vx,
    "coef_err_vy": coef_vy,
})
coef_tbl["abs_sum"] = coef_tbl["coef_err_vx"].abs() + coef_tbl["coef_err_vy"].abs()
coef_tbl.sort_values("abs_sum", ascending=False).head(25)

## Predicted Velocity Curves vs OptiTrack

The plots below show a single recording in time domain:

- `OptiTrack true` (robot-frame truth from transformed OptiTrack velocity)
- `Measured` (raw `sport_vx/sport_vy`)
- `Predicted` (measured + polynomial error estimate)


In [ ]:
# Pick a representative recording (most samples).
rec_counts = data.groupby("recording").size().sort_values(ascending=False)
demo_rec = rec_counts.index[0]
demo = data[data["recording"] == demo_rec].copy().reset_index(drop=True)

X_demo = demo[feature_cols].to_numpy()
pred_err_vx_demo = model_vx.predict(X_demo)
pred_err_vy_demo = model_vy.predict(X_demo)

vx_true_demo = demo["true_vx"].to_numpy()
vy_true_demo = demo["true_vy"].to_numpy()
vx_meas_demo = demo["imu_vx"].to_numpy()
vy_meas_demo = demo["imu_vy"].to_numpy()
vx_pred_demo = vx_meas_demo + pred_err_vx_demo
vy_pred_demo = vy_meas_demo + pred_err_vy_demo

t_demo = np.cumsum(demo["dt"].to_numpy())
t_demo -= t_demo[0]

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle(f"Velocity Curves - recording: {demo_rec}", fontsize=12)

axes[0].plot(t_demo, vx_true_demo, label="vx OptiTrack true", linewidth=2.0)
axes[0].plot(t_demo, vx_meas_demo, label="vx measured", linewidth=1.3, alpha=0.85)
axes[0].plot(t_demo, vx_pred_demo, label="vx predicted", linewidth=1.5)
axes[0].set_ylabel("vx [m/s]")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(t_demo, vy_true_demo, label="vy OptiTrack true", linewidth=2.0)
axes[1].plot(t_demo, vy_meas_demo, label="vy measured", linewidth=1.3, alpha=0.85)
axes[1].plot(t_demo, vy_pred_demo, label="vy predicted", linewidth=1.5)
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("vy [m/s]")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

rmse_vx_meas = np.sqrt(np.mean((vx_true_demo - vx_meas_demo) ** 2))
rmse_vx_pred = np.sqrt(np.mean((vx_true_demo - vx_pred_demo) ** 2))
rmse_vy_meas = np.sqrt(np.mean((vy_true_demo - vy_meas_demo) ** 2))
rmse_vy_pred = np.sqrt(np.mean((vy_true_demo - vy_pred_demo) ** 2))

print(f"Demo recording: {demo_rec}")
print(f"vx RMSE measured -> predicted: {rmse_vx_meas:.5f} -> {rmse_vx_pred:.5f} m/s")
print(f"vy RMSE measured -> predicted: {rmse_vy_meas:.5f} -> {rmse_vy_pred:.5f} m/s")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle("Polynomial error model: predicted vs true error", fontsize=13)

# err_vx scatter
limx = np.quantile(np.abs(np.r_[y_vx, pred_err_vx]), 0.995)
axes[0, 0].scatter(y_vx, pred_err_vx, s=5, alpha=0.25)
axes[0, 0].plot([-limx, limx], [-limx, limx], 'r--', lw=1.0)
axes[0, 0].set_title("err_vx")
axes[0, 0].set_xlabel("true error")
axes[0, 0].set_ylabel("predicted error")
axes[0, 0].grid(True, alpha=0.3)

# err_vy scatter
limy = np.quantile(np.abs(np.r_[y_vy, pred_err_vy]), 0.995)
axes[0, 1].scatter(y_vy, pred_err_vy, s=5, alpha=0.25)
axes[0, 1].plot([-limy, limy], [-limy, limy], 'r--', lw=1.0)
axes[0, 1].set_title("err_vy")
axes[0, 1].set_xlabel("true error")
axes[0, 1].set_ylabel("predicted error")
axes[0, 1].grid(True, alpha=0.3)

# residual histograms
res_vx = y_vx - pred_err_vx
res_vy = y_vy - pred_err_vy
axes[1, 0].hist(res_vx, bins=60, alpha=0.7)
axes[1, 0].set_title("residual err_vx")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(res_vy, bins=60, alpha=0.7)
axes[1, 1].set_title("residual err_vy")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
out = {
    "poly_degree": np.array([POLY_DEGREE]),
    "ridge_alpha": np.array([RIDGE_ALPHA]),
    "feature_names": np.array(feat_names, dtype=object),
    "coef_err_vx": coef_vx,
    "coef_err_vy": coef_vy,
    "intercept_err_vx": np.array([int_vx]),
    "intercept_err_vy": np.array([int_vy]),
}

save_path = Path("go2_velocity_error_poly_model.npz")
np.savez(save_path, **out)
print(f"Saved model parameters to: {save_path}")